This notebook computes the statisics shown in the paper regarding the optimal RAG configuration.

In [5]:
# This line is needed to be able to import functions from the pr
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))

import pandas as pd
from IPython.display import display, HTML
from plot_results import format_experiment_data, load_statistics

In [6]:
### Here are the hyperparameters of this analysis

# Where the results are stored
data_root = "../data/results"

# The dataset name
dataset_name = "nqopen_small_test"


In [7]:
# Path to the root of the benchmark results
root_benchmark_path = f'{data_root}/{dataset_name}/optimal_configuration/'

def load_results(root_benchmark_path):
        
    assert os.path.exists(root_benchmark_path), f"The path {root_benchmark_path} does not exist. Please make sure that you set correctly the data_root and dataset_name variables."

    # Check if a data/ folder already exists
    if not os.path.exists(f"{data_root}/{dataset_name}/data"):
        
        # If the data folder does not exis, the benchmark results are still in raw format and we should process them before the analysis
        format_experiment_data(root_benchmark_path)

    # Load all statistics in a single dataframe
    df_results_all = load_statistics(root_benchmark_path)

    # Convert 'Ben','Mal','Ben&mal','Hallucination','Avg # mal docs in prompt' to numeric
    df_results_all['Ben'] = pd.to_numeric(df_results_all['Ben'])
    df_results_all['Mal'] = pd.to_numeric(df_results_all['Mal'])
    df_results_all['Ben&mal'] = pd.to_numeric(df_results_all['Ben&mal'])
    df_results_all['Hallucination'] = pd.to_numeric(df_results_all['Hallucination'])
    df_results_all['Avg # mal docs in prompt'] = pd.to_numeric(df_results_all['Avg # mal docs in prompt'])

    # Set Injection strategy to null if nan
    df_results_all.loc[df_results_all['Injection strategy']=='nan','Injection strategy'] = ''
    
    return df_results_all
df_results = load_results(root_benchmark_path)


Here follows a table describing the performance of the optimized pipeline against different attacks scenarios.

In [8]:
df_default = df_results[df_results['Parameter']=='default']
df_optimal = df_results[df_results['Parameter']=='optimal']

# merge df_default and df_optimal on the column 'Optimization' adding a suffix the suffix '_opt' to the columns of df_optimal
df = pd.merge(df_default, df_optimal, on='Optimization', suffixes=('', '_opt'))

display(HTML(df[['Parameter','Optimization','Ben','Ben_opt','Mal','Mal_opt','Ben&mal','Ben&mal_opt','Hallucination','Hallucination_opt']].to_html(index=False)))

Parameter,Optimization,Ben,Ben_opt,Mal,Mal_opt,Ben&mal,Ben&mal_opt,Hallucination,Hallucination_opt
default,unoptimized,0.51,0.57,0.11,0.07,0.18,0.11,0.20,0.25
default,query+,0.46,0.62,0.25,0.06,0.18,0.08,0.11,0.24
default,seo-documents-writer,0.44,0.62,0.31,0.07,0.15,0.06,0.10,0.25
default,IDEM,0.33,0.62,0.23,0.06,0.22,0.12,0.22,0.20
default,PoisonRAG-LM-targeted,0.44,0.71,0.34,0.03,0.18,0.05,0.04,0.21
default,query+answer,0.56,0.67,0.19,0.04,0.11,0.05,0.14,0.24
default,Phantom-corruption,0.25,0.73,0.62,0.01,0.00,0.03,0.13,0.23
default,PAT,0.51,0.56,0.13,0.06,0.18,0.13,0.18,0.25
default,asc-natural-noreg,0.65,0.63,0.11,0.06,0.09,0.04,0.15,0.27
